In [7]:
!pip install beautifulsoup4 sentence-transformers requests

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [requests]


In [1]:
from details import initial_sources
from bs4 import BeautifulSoup
import requests
import urllib
import re
from enum import Enum
from sentence_transformers import SentenceTransformer
import traceback

/Users/karankamat/Desktop/manchester/TTM/coursework/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import details
import time
import json

In [3]:
#model=SentenceTransformer("all-MiniLM-L6-v2")
model=SentenceTransformer("all-mpnet-base-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7469.45it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
#from details import original_concepts
original_concepts=['South Asian Cuisine',
    'Indian_cuisine',
    'Pakistani_cuisine',
    'Bangladeshi_cuisine',
    'Sri_Lankan_cuisine',
    'Nepalese_cuisine',
    'Afghhani Cuisine',
    'Bengali Cuisine',
    ]
original_concept_embedding=model.encode(original_concepts,convert_to_tensor=True)

In [ ]:
def is_south_asian_cuisine(text):
    text=text.lower()
    geographic_terms = [
    # Countries & adjective forms
    "india", "pakistan", "bangladesh", 
    "sri lanka", "nepal","bhutan",
    "maldives", "afghan", "bengal", "south asian", "subcontinent",
    "sylheti",

    # Key subregions
    "punjab", "gujarat", "rajasthan", "kashmir", "hyderabad",
    "assam", "awadh", "tamil", "kerala", "sindh",

    # Major cities
    "lucknow", "kolkata", "mumbai", "delhi", "lahore", "karachi", "dhaka",
]

    food_terms = [
    # Unique ingredients & techniques
    "ghee", "paneer", " dal", "dahi", "tadka", "biryani", "chaat",
    "samosa", "roti", "naan", "dosa", "idli", "korma", "saag",
    "garam masala", "tamarind", "asafoetida", "besan",
    "kadai", "tandoor", "tawa", "mithai", " lassi","lassi "
]

#     negative_terms = [
#     "hummus", "pita", "tahini", "meze",           # Middle Eastern
#     "lemongrass", "fish sauce", "galangal",        # Southeast Asian
#     "soy sauce", "miso", "dashi",                  # East Asian
#     "harissa", "couscous", "olive oil",            # North African / Mediterranean
# ]
#     for substring in negative_terms:
#         if substring in text:
#             return (False,'')
    for substring in geographic_terms:
        if substring in text:
            return (True,substring)
    for substring in food_terms:
        if substring in text:
            return (True,substring)
    
    return (False,'')


In [ ]:
import json
with open('south_asian_corpus.json','r') as file:
    k=json.load(file)


327

In [6]:
import sentence_transformers.util
import torch

In [7]:
class Section:
    section_title:str
    section_text:str
    def __init__(self,title) -> None:
        self.section_text=""
        self._child_count=0
        self.section_title=title
    def to_dict(self):
        return {"section_title":self.section_title,"section_text":self.section_text}
    def is_south_asian_cuisine(self):
         is_sac,value=is_south_asian_cuisine(self.section_title)
         if is_sac:
             return (is_sac,value)
         is_sac,value= is_south_asian_cuisine(self.section_text)
         if is_sac:
             return (is_sac,value)
         return (False,'')

class SiteType(Enum):
    WIKIPEDIA=1
    WIKIBOOK=2
    CUISINES80=3
    ERROR=4

class RelevantURL:
    url:str
    relevance:float
    concept: str
    source: str
    def __init__(self,concept,url,relevance,source) -> None:
        self.url=url
        self.relevance=relevance
        self.concept=concept
        self.source=source
    def __repr__(self):
        return f"{self.concept} - {self.relevance}                  {self.url}.   from {self.source}"
    def to_dict(self):
        return {"concept":self.concept,"relevance":self.relevance,"url":self.url}





In [8]:
class Website:
    number_of_sites:int=0
    doc_number:str
    url:str
    title:str
    type:SiteType
    document:BeautifulSoup
    list_of_urls:list[str]
    relevant_suburls:list[RelevantURL]
    categories:list[str]
    content:list[Section]
    references:Section
    summary:Section
    has_been_extracted:bool=False
    title_relevance:float
    categorical_relevance:float
    summary_relevance:float
    site_source:str
    simple_binary_south_asian_cuisine_check:bool
    simple_value_south_asian_cuisine:str
    def __init__(self,url_value,email_address="karan.kamat@postgrad.manchester.ac.uk",extract_doc=True,title_relevance=0.0,site_source='') -> None:
        self.title_relevance=title_relevance
        self.url=url_value
        self.summary=Section("summary")
        self.site_source=site_source
        if "wikipedia" in url_value:
            self.type=SiteType.WIKIPEDIA
            self.title=Website.url_title_extraction(self.url,SiteType.WIKIPEDIA)
        elif ("wikibooks" in url_value) and ("Cookbook"in url_value or "Category" in url_value):
            self.type=SiteType.WIKIBOOK
        elif "aroundtheworldin80cuisinesblog" in url_value:
            self.type=SiteType.CUISINES80
        else:
            self.type=SiteType.ERROR
            return
        if extract_doc:
            self._document_extraction(email_address)

    @staticmethod
    def url_title_extraction(url,type):
        if type==SiteType.WIKIPEDIA or type==SiteType.WIKIBOOK:
            url=url[url.find("/wiki/"):]
            cleaned = url.replace('/wiki/', '').replace('_', ' ').strip()
            cleaned=urllib.parse.unquote(cleaned)
            cleaned = re.sub(r'\s+', ' ', cleaned)
            if type==SiteType.WIKIBOOK and ':' in cleaned:
                cleaned=cleaned[cleaned.find(":")+1:]
        return cleaned
        
    def to_dict(self)->dict:
        return {"url":self.url,
            "title":self.title,
            "type":self.type.name,
            "categories":self.categories,
            #"list_of_urls":self.list_of_urls,
            #"relevant_suburls":[relevant_url.to_dict() for relevant_url in self.relevant_suburls],
            "summary":self.summary.to_dict(),
            #"references":self.references.to_dict(),
            "content":[section.to_dict() for section in self.content],
            # "categorical_relevance":self.categorical_relevance,
            # "summary_relevance":self.summary_relevance,
            # "title_relevance":self.title_relevance,
            # "total_relevance":self.total_relevance,
            "site_source":self.site_source
        }

    def is_site_south_asian_cuisine_simple(self):
        if is_south_asian_cuisine(self.title)[0]:
            self.simple_binary_south_asian_cuisine_check=True
            self.simple_value_south_asian_cuisine=is_south_asian_cuisine(self.title)[1]
            return True
        for category in self.categories:
            if is_south_asian_cuisine(category)[0]:
                self.simple_binary_south_asian_cuisine_check=True
                self.simple_value_south_asian_cuisine=is_south_asian_cuisine(category)[1]
                return True
        for section in self.content:
            if section.is_south_asian_cuisine()[0]:
                self.simple_binary_south_asian_cuisine_check=True
                self.simple_value_south_asian_cuisine=section.is_south_asian_cuisine()[1]
                return True
        if is_south_asian_cuisine(self.site_source)[0]:
            self.simple_binary_south_asian_cuisine_check=True
            self.simple_value_south_asian_cuisine=is_south_asian_cuisine(self.site_source)[1]
            return True
        self.simple_binary_south_asian_cuisine_check=False
        self.simple_value_south_asian_cuisine=''
        return False

    def document_title_extraction(self):
        if self.type==SiteType.WIKIPEDIA:
            return self.title
        target_divs=self.document.find_all('title')#,class_='mw-page-title-main')
        if target_divs:
            for value in target_divs:
                title=value.text
                title=title[:title.find(" - Wiki")]
        if self.type==SiteType.WIKIBOOK and ':' in title:
            title=title[title.find(":")+1:]
        self.title=title


    def _document_extraction(self,email_address):
        headers = {f'User-Agent': f'Cuisine student project ({email_address})  Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
        if self.type==SiteType.WIKIBOOK:
            response = requests.get(self.url, headers=headers)
            if response.status_code !=200:
                print("Failed to retrieve page - status code",response.status_code)
            else:
                doc=BeautifulSoup(response.text,'html.parser')
                self.document=doc
        elif self.type==SiteType.WIKIPEDIA:
            pass
        return response.status_code
    
    def document_all_urls_extraction(self):
        acceptable_classes=["mw-content-ltr mw-parser-output","mw-category mw-category-columns","mw-category"]
        list_of_urls=[]
        for acceptable_class in acceptable_classes:
            target_div=self.document.find_all('div',class_=acceptable_class)
            if target_div:
                for link in target_div[0].find_all('a',href=True):
                    url:str=str(link["href"])
                    if self.type==SiteType.WIKIBOOK:
                        if url.startswith("/wiki/") and 'File' not in url:
                            list_of_urls.append("https://en.wikibooks.com"+url)
                    elif self.type==SiteType.WIKIPEDIA:
                        if url.startswith("/wiki/") and 'File' not in url:
                            list_of_urls.append('https://en.wikipedia.org/'+url)
        self.list_of_urls=list_of_urls
        return
    
    def categorical_relevance_calculation(self):
        all_categories=self.categories.copy()
        if len(all_categories)==0:
            self.categorical_relevance=-1
            return
        new_categories_embedding=model.encode(all_categories,convert_to_tensor=True)
        scores=sentence_transformers.util.pytorch_cos_sim(original_concept_embedding,new_categories_embedding)
        scores=torch.max(scores,dim=0)
        scores=[float(score.item()) for score in scores[0]]
        scores.sort(reverse=True)
        scores=scores[:3] if len(scores)>3 else scores
        self.categorical_relevance=sum(scores)/len(scores)
        return

    def summary_relevance_calculation(self):
        try:
            if self.summary!="":
                summary_embedding=model.encode([self.summary.section_text],convert_to_tensor=True)
                scores=sentence_transformers.util.pytorch_cos_sim(original_concept_embedding,summary_embedding)
                self.summary_relevance=torch.max(scores,dim=0)[0][0].item()
            else:
                self.summary_relevance=-1
        except:
            self.summary_relevance=-1

    def document_relevant_urls_extraction(self):
        relevant_urls=[]
        if self.type==SiteType.WIKIBOOK:
            # original_concept=['South Asian Cuisine']
            if len(self.list_of_urls)==0:
                self.relevant_suburls=[]
                return []
            list_of_concepts=['']*len(self.list_of_urls)
            for i,url in enumerate(self.list_of_urls):
                title=Website.url_title_extraction(url,self.type)
                list_of_concepts[i]=title
            new_concepts_embedding=model.encode(list_of_concepts,convert_to_tensor=True)
            scores=sentence_transformers.util.pytorch_cos_sim(original_concept_embedding,new_concepts_embedding)
            scores=torch.max(scores,dim=0)
            relevant_urls=[RelevantURL(concept,url,float(score.item()),self.title) for concept,url,score in zip(list_of_concepts,self.list_of_urls,scores[0])]
            relevant_urls.sort(key=lambda x:x.relevance)
            self.relevant_suburls=relevant_urls
        elif self.type==SiteType.WIKIPEDIA:
            pass
        return relevant_urls
    
    def document_categories_extraction(self):
        if self.type in [SiteType.WIKIBOOK,SiteType.WIKIPEDIA]:
            categories=[]
            target_div=self.document.find('div',id='mw-normal-catlinks')
            if target_div:
                urls=target_div.find_all('a',title=True)
                for url in urls:
                    full_category:str=str(url.get('title'))
                    if full_category.startswith('Category:'):
                        category=full_category[9:]
                        categories.append(category)
            self.categories=categories
    
    def document_content_extraction(self):
        target_div=self.document.find_all('div',class_="mw-content-ltr mw-parser-output")[0]
        self.content=[]
        section=Section("Introduction")
        section_started=False
        if self.type==SiteType.WIKIBOOK:
            for child in target_div.children:
                # print(child)
                # print(child.text)
                if str(child.text).startswith(".mw-parser-output") or str(child.text).startswith("Cookbook | Recipes"):
                    # print(f"Problem-child --------{child}")
                    continue
                if str(child).startswith('<div class="mw-heading mw-heading2">'):
                    section_started=True
                    self.content.append(section)
                    if section.section_title=="Introduction":
                        self.summary=section
                    child_title=child.text
                    child_title=child_title[:child_title.find("[edit")]
                    section=Section(child_title)
                elif section_started:
                    section_started=False
                    section.section_text=child.text+"\n"
                else:
                    section.section_text=section.section_text+child.text+"\n"
            if section.section_text!="" :
                if section.section_title=="References":
                    self.references=section
                else:
                    self.content.append(section)
                    self.references=Section("References")
                    if section.section_title=="Introduction":
                        self.summary=section

        return self.content

    def total_relevance_calculation(self):
        sum_rel=self.summary_relevance
        cat_rel=self.categorical_relevance
        title_rel=self.title_relevance
        total_rel:float
        if sum_rel==-1 and cat_rel==-1:
            total_rel=title_rel
        elif sum_rel==-1:
            total_rel=cat_rel*0.6+title_rel*.4
        elif cat_rel==-1:
            total_rel=sum_rel*0.7+title_rel*.3
        else:
            total_rel=sum_rel*0.5+cat_rel*0.3+title_rel*0.2
        self.total_relevance=total_rel
                    
        
    def extract_all(self):
        if self.has_been_extracted==True:
            return
        self.document_all_urls_extraction()
        self.document_categories_extraction()
        self.document_title_extraction()
        self.document_content_extraction()
        self.has_been_extracted=True
    
    def calculate_relevance(self):
        if not self.has_been_extracted:
            return
        self.categorical_relevance_calculation()
        self.summary_relevance_calculation()
        self.total_relevance_calculation()
        self.document_relevant_urls_extraction()

In [29]:
# def lets_go_bros():
    # sitea=Website()
with open("saved_wikibook_data.json",'r') as json_file:
    prev_simple_corpus=json.load(json_file)
    title_list=set(c['title'] for c in prev_simple_corpus)
simple_corpus=[]
yes_titles=[]
no_titles=[]
with open("saved_wikibook_simple_data.json","w") as json_file:
    try:
        initial=RelevantURL(url="https://en.wikibooks.org/wiki/Cookbook:South_Asian_Cuisine",relevance=1.0,concept="South Asian Cuisine",source="ROOT/SELF")
        all_rel_links=set()
        link_queue=[initial]
        count=1
        while(len(link_queue)!=0) and count<500:
            time.sleep(0.3)
            link=link_queue.pop(0)
            if link.url in all_rel_links:
                if not (link.url in no_titles and is_south_asian_cuisine(link.source)):
                    continue
            print(link)
            all_rel_links.add(link.url)
            site=Website(link.url,title_relevance=link.relevance,site_source=link.source)
            if site.type==SiteType.ERROR:
                continue
            site.extract_all()
            print(site.title,site.is_site_south_asian_cuisine_simple(),site.simple_value_south_asian_cuisine)
            if site.is_site_south_asian_cuisine_simple():
                yes_titles.append(link.url)
                count+=1
                simple_corpus.append(site.to_dict())
                site.document_relevant_urls_extraction()
                print(count)
                link_queue.extend(site.relevant_suburls)
            else:
                no_titles.append(link.url)
        json.dump(simple_corpus,json_file)
    except Exception as e:
        tb_str = "".join(traceback.format_exception(type(e), e, e.__traceback__))
        print(tb_str)
        json.dump(simple_corpus,json_file)
l=list(all_rel_links)

South Asian Cuisine - 1.0                  https://en.wikibooks.org/wiki/Cookbook:South_Asian_Cuisine.   from ROOT/SELF
South Asian Cuisine True south asian
2
Sweet Lassi - 0.270754873752594                  https://en.wikibooks.com/wiki/Cookbook:Sweet_Lassi.   from South Asian Cuisine
Sweet Lassi True  lassi
3
Mild Salty Lassi - 0.30186936259269714                  https://en.wikibooks.com/wiki/Cookbook:Mild_Salty_Lassi.   from South Asian Cuisine
Mild Salty Lassi True  lassi
4
Phulourie (Split Pea Fritters) - 0.3088245987892151                  https://en.wikibooks.com/wiki/Cookbook:Phulourie_(Split_Pea_Fritters).   from South Asian Cuisine
Phulourie (Split Pea Fritters) True india
5
Sylheti Handesh - 0.3572484254837036                  https://en.wikibooks.com/wiki/Cookbook:Sylheti_Handesh.   from South Asian Cuisine
Sylheti Handesh True sylheti
6
Ingredients - 0.38970234990119934                  https://en.wikibooks.com/wiki/Category:Ingredients.   from South Asian Cuisine
Ingredi

Only tests below this threshold - No need to run

In [27]:
set(link.url[link.url.find("/wiki/"):] for link in link_queue[:300] if link.url not in yes_titles)

{'/wiki/Category:Appliances',
 '/wiki/Category:Bakeware',
 '/wiki/Category:Cookbook_equipment',
 '/wiki/Category:Cookbook_pages_needing_work',
 '/wiki/Category:Cookbook_units',
 '/wiki/Category:Dairy',
 '/wiki/Category:Extracts_and_flavorings',
 '/wiki/Category:Featured_ingredients',
 '/wiki/Category:Featured_recipe_candidates',
 '/wiki/Category:Featured_recipes',
 '/wiki/Category:Kid-friendly_recipes',
 '/wiki/Category:Recipes_by_diet',
 '/wiki/Category:Recipes_by_difficulty',
 '/wiki/Category:Recipes_by_expense',
 '/wiki/Category:Recipes_by_ingredient',
 '/wiki/Category:Recipes_by_meal_or_course',
 '/wiki/Category:Recipes_by_occasion',
 '/wiki/Category:Recipes_by_origin',
 '/wiki/Category:Recipes_by_preparation_technique',
 '/wiki/Category:Recipes_by_source',
 '/wiki/Category:Recipes_by_type',
 '/wiki/Category:Recipes_with_images',
 '/wiki/Category:Recipes_with_metric_units',
 '/wiki/Category:Utensils',
 '/wiki/Cookbook:%C3%87%C4%B1lb%C4%B1r_(Turkish_Eggs_with_Yogurt)',
 '/wiki/Cookb

In [38]:
site=Website("https://en.wikibooks.org/wiki/Cookbook:Saffron")
site.extract_all()

In [39]:
site.is_site_south_asian_cuisine_simple()

False

In [30]:
if is_south_asian_cuisine(site.title)[0]:
    site.simple_binary_south_asian_cuisine_check=True
    site.simple_value_south_asian_cuisine=is_south_asian_cuisine(site.title)[1]
    print(True,1)
else:
    print(site.title)
for category in site.categories:
    if is_south_asian_cuisine(category)[0]:
        site.simple_binary_south_asian_cuisine_check=True
        site.simple_value_south_asian_cuisine=is_south_asian_cuisine(category)[1]
        print(True,2)
print(site.categories)
for section in site.content:
    if section.is_south_asian_cuisine()[0]:
        site.simple_binary_south_asian_cuisine_check=True
        site.simple_value_south_asian_cuisine=section.is_south_asian_cuisine()[1]
        print(True,3)

True 1
True 2
['Medium Difficulty recipes', 'Recipes', 'Snack recipes', 'Recipes using rice', 'Sylheti recipes', 'Recipes for fritters', 'Fried recipes', 'Recipes using rice flour', 'Recipes using fresh ginger']
True 3
True 3


(True, 'sylheti')

In [12]:
print(*[title[title.find("/wiki/")+5:] for title in yes_titles],sep='\n')

/Cookbook:South_Asian_Cuisine
/Cookbook:Sweet_Lassi
/Cookbook:Mild_Salty_Lassi
/Cookbook:Phulourie_(Split_Pea_Fritters)
/Cookbook:Sylheti_Handesh
/Category:Ingredients
/Cookbook:Papadam_(Black_Gram_Flatbread)
/Cookbook:Papaya_Lassi
/Cookbook:Sweet_Mango_Lassi
/Cookbook:Mango_Chutney_(Chunky)
/Cookbook:Mango_Chutney_(Smooth)
/Cookbook:Sylheti_Doner_Kebab
/Category:Cookbook
/Category:Sylheti_recipes
/Cookbook:Masyaura_(Nepali_Fermented_Vegetable_Balls)
/Cookbook:Masala_Chai_II
/Cookbook:Sylheti_Rice_Pudding
/Cookbook:Malpua_(South_Asian_Sweet_Pancake)
/Cookbook:Seviyan_Ji_Khirni_(Sindhi_Vermicelli_Pudding)
/Cookbook:Dum_ka_Qimah_(Spiced_Minced_Meat)
/Cookbook:Corom_Chatni_(Mango_Chutney_with_Hot_Chillies)
/Cookbook:Prawn_Curry
/Cookbook:Khatti_Dal_(Spiced_Tamarind_Pigeon_Peas)
/Cookbook:Qabuli_(Central_Asian_Rice_Pilaf)
/Cookbook:Watalappam_(Sri_Lankan_Coconut_Custard)
/Cookbook:Papri_Chaat_(Crispy_Indian_Snack_with_Potato)
/Category:Tibetan_recipes
/Cookbook:Sindhi_Fried_Potatoes
/Cookb

In [13]:
print(*[title[title.find("/wiki/")+5:] for title in no_titles],sep='\n')

/Cookbook:Paper_Towel
/Cookbook:Inch
/Cookbook:Beating
/Cookbook:Pound
/Cookbook:Pint
/Cookbook:Egg
/Category:Fritter_recipes
/Cookbook:Oil
/Cookbook:Pea
/Cookbook:Food_Processor
/Cookbook:Frying
/Cookbook:Frying
/Cookbook:Garlic
/Cookbook:Turmeric
/Cookbook:Flour
/Cookbook:Paper_Towel
/Cookbook:Date_Syrup
/Category:Recipes_for_dessert
/Cookbook:Kneading
/Cookbook:Dough
/Cookbook:Allspice
/Cookbook:Cloudberry
/Cookbook:Blood
/Cookbook:Carbonated_Water
/Cookbook:Cherry
/Cookbook:Akamu_Paste
/Cookbook:Barberry
/Cookbook:Conch
/Cookbook:Chervil
/Cookbook:Crouton
/Cookbook:Caraway
/Cookbook:Caper
/Cookbook:Cornelian_Cherry
/Cookbook:Cognac
/Cookbook:Coffee_Liqueur
/Cookbook:Baker%27s_Ammonia
/Cookbook:Brandy
/Cookbook:Condensed_Milk
/Cookbook:Baking_Soda
/Cookbook:Chanterelle
/Cookbook:Cracker
/Cookbook:Aquafaba
/Cookbook:Aj%C3%AD_Dulce
/Cookbook:Cod
/Cookbook:Apple
/Cookbook:Aspartame
/Cookbook:Chestnut
/Cookbook:Bison
/Cookbook:Alfalfa_Sprouts
/Cookbook:Clotted_Cream
/Cookbook:Cherimoya


In [ ]:
# def lets_go_bros():
    # sitea=Website()
with open("saved_wikibook_data.json",'r') as json_file:
    prev_corpus=json.load(json_file)
    title_list=set(c['title'] for c in prev_corpus)
with open("saved_wikibook_data.json","w") as json_file:
    try:
        initial=RelevantURL(url="https://en.wikibooks.org/wiki/Cookbook:South_Asian_Cuisine",relevance=1.0,concept="South Asian Cuisine",source="ROOT/SELF")
        all_rel_links=set()
        link_queue=[initial]
        count=1
        while(len(link_queue)!=0) and count<1000:
            link=link_queue.pop(0)
            print(link)
            if link.url in all_rel_links:
                continue
            all_rel_links.add(link.url)
            if link.relevance<0.2:
                continue
            site=Website(link.url,title_relevance=link.relevance,site_source=link.source)
            count+=1
            site.extract_all()
            if site.title not in title_list:
                site.calculate_relevance()
                print(f"==== category:{site.categorical_relevance}, title:{site.title_relevance}, summary:{site.summary_relevance}    total ={site.total_relevance}.    source={site.site_source}")
                prev_corpus.append(site.to_dict())
            else:
                site.document_relevant_urls_extraction()
            print(count)
            link_queue.extend(site.relevant_suburls)
        json.dump(prev_corpus,json_file)
    except Exception as e:
        print(e)
        json.dump(prev_corpus,json_file)
l=list(all_rel_links)

South Asian Cuisine - 1.0                  https://en.wikibooks.org/wiki/Cookbook:South_Asian_Cuisine.   from ROOT/SELF
2
Sweet Lassi - 0.270754873752594                  https://en.wikibooks.com/wiki/Cookbook:Sweet_Lassi.   from South Asian Cuisine
3
Mild Salty Lassi - 0.30186936259269714                  https://en.wikibooks.com/wiki/Cookbook:Mild_Salty_Lassi.   from South Asian Cuisine
4
Phulourie (Split Pea Fritters) - 0.3088245987892151                  https://en.wikibooks.com/wiki/Cookbook:Phulourie_(Split_Pea_Fritters).   from South Asian Cuisine
5
Sylheti Handesh - 0.3572484254837036                  https://en.wikibooks.com/wiki/Cookbook:Sylheti_Handesh.   from South Asian Cuisine
6
Ingredients - 0.38970234990119934                  https://en.wikibooks.com/wiki/Category:Ingredients.   from South Asian Cuisine
7
Papadam (Black Gram Flatbread) - 0.4123512804508209                  https://en.wikibooks.com/wiki/Cookbook:Papadam_(Black_Gram_Flatbread).   from South Asian Cuisine

[{'section_title': 'Introduction',
  'section_text': 'Puttu (Steamed Rice Flour and Coconut) CategoryIndian recipesDifficulty\n\n\nPuttu is a breakfast dish in Kerala, India. The main ingredients are rice flour and coconut. It is usually eaten with steamed bananas or Kadala Curry (chickpea curry), Chicken Curry, or boiled green gram (moong) and pappad.\n\n\n\n'},
 {'section_title': 'Ingredients',
  'section_text': '\n\n1 medium-sized coconut\n4 cups puttu powder (rice flour)\nSalt to taste\nWater (enough to moisten the flour)\n\n\n'},
 {'section_title': 'Procedure',
  'section_text': '\n\nGrate the coconut flesh finely.\nMix the puttu powder and salt in a wide bowl. Add half of the grated coconut.\nRub the mixture with your fingertips, sprinkling in water until it is moist and develops a coarse texture.\nLeave the mixture for an hour if the flour is ready made, as it absorbs more water.\nCheck the consistency of the flour. One should be able to make a ball out of the flour that can be 

In [255]:
len(sac_not)/len(corpus)*100

49.5118549511855

[]

In [81]:
len(set(c['url'] for c in corpus))

533

In [15]:
siteS=Website("https://en.wikibooks.org/wiki/Cookbook:South_Asian_Cuisine")
siteS.extract_all()

In [85]:
sum([c['categorical_relevance']==0 for c in corpus])

155

In [350]:
sitea.type.name

'WIKIBOOK'

In [ ]:
siteb=Website("https://en.wikibooks.org/wiki/Cookbook:South_Asian_Cuisine")
sitec=Website("https://en.wikibooks.org/wiki/Cookbook:Knife_Skills#Chopping")

[('https://en.wikibooks.com/wiki/Category:Cookbook',
  tensor([0.3409, 0.2731, 0.6472, 0.8984, 0.4349, 0.4758, 0.5881, 0.4692, 0.5343,
          0.3749, 0.4699, 1.0000, 0.4012, 0.4654, 0.4673, 0.5538, 0.3928, 0.3685,
          0.2994, 0.4224, 0.2106, 0.3943, 0.2914, 0.4344, 0.0936, 0.2936, 0.4814,
          0.5355, 0.4501, 0.1253, 0.3388, 0.1749, 0.5094, 0.2151, 0.5166, 0.4110],
         device='mps:0'))]

In [ ]:
original_concept=['South Asian Cuisine',
    'Indian_cuisine',
    'Pakistani_cuisine',
    'Bangladeshi_cuisine',
    'Sri_Lankan_cuisine',
    'Nepalese_cuisine',
    'Afghhani Cuisine',
    'Bengali Cuisine',
    ]
extra_concepts=["USA","Marker","Table","Jacket","Machine Learning","Whatsapp","Macbook","University of Manchester","Jimmy","Barcelona","Drinking Water"]
original_concept_embedding=model.encode(original_concept,convert_to_tensor=True)
new_concepts_embedding=model.encode(list_of_concepts+extra_concepts,convert_to_tensor=True)

In [292]:
scores=sentence_transformers.util.pytorch_cos_sim(original_concept_embedding,new_concepts_embedding)
scores=torch.max(scores,dim=0)

In [293]:
relevant_urls=[(concept,score) for concept,score in zip(list_of_concepts+extra_concepts,scores[0])]
relevant_urls.sort(key=lambda x:x[1].item(),reverse=True)
relevant_urls

[('South Asian Cuisine', tensor(1., device='mps:0')),
 ('Bengali recipes', tensor(0.8982, device='mps:0')),
 ('Asian Cuisine', tensor(0.8507, device='mps:0')),
 ('Pakistani recipes', tensor(0.7878, device='mps:0')),
 ('Nepali recipes', tensor(0.7422, device='mps:0')),
 ('Cuisines', tensor(0.7031, device='mps:0')),
 ('Indian recipes', tensor(0.6994, device='mps:0')),
 ('Afghan recipes', tensor(0.6607, device='mps:0')),
 ('Sylheti Fried Rice', tensor(0.6258, device='mps:0')),
 ('Sindhi Fried Potatoes', tensor(0.6174, device='mps:0')),
 ('Tibetan recipes', tensor(0.6151, device='mps:0')),
 ('Papri Chaat (Crispy Indian Snack with Potato)',
  tensor(0.5939, device='mps:0')),
 ('Watalappam (Sri Lankan Coconut Custard)', tensor(0.5862, device='mps:0')),
 ('Qabuli (Central Asian Rice Pilaf)', tensor(0.5827, device='mps:0')),
 ('Khatti Dal (Spiced Tamarind Pigeon Peas)', tensor(0.5771, device='mps:0')),
 ('Prawn Curry', tensor(0.5732, device='mps:0')),
 ('Corom Chatni (Mango Chutney with Hot Ch

In [258]:
len(list_of_concepts),len(scores[0])

(36, 36)

In [255]:
type(relevant_urls[0][1])

torch.Tensor

In [249]:
lou=siteb.list_of_urls
list_of_concepts=['']*len(lou)
for i,url in enumerate(lou):
    title=Website.url_title_extraction(url,SiteType.WIKIBOOK)
    list_of_concepts[i]=title
list_of_concepts

['Cookbook',
 'Ingredients',
 'Cuisines',
 'Asian Cuisine',
 'Afghan recipes',
 'Bengali recipes',
 'Indian recipes',
 'Nepali recipes',
 'Pakistani recipes',
 'Sylheti recipes',
 'Tibetan recipes',
 'South Asian Cuisine',
 'Corom Chatni (Mango Chutney with Hot Chillies)',
 'Dum ka Qimah (Spiced Minced Meat)',
 'Khatti Dal (Spiced Tamarind Pigeon Peas)',
 'Malpua (South Asian Sweet Pancake)',
 'Mango Chutney (Chunky)',
 'Mango Chutney (Smooth)',
 'Masala Chai II',
 'Masyaura (Nepali Fermented Vegetable Balls)',
 'Mild Salty Lassi',
 'Papadam (Black Gram Flatbread)',
 'Papaya Lassi',
 'Papri Chaat (Crispy Indian Snack with Potato)',
 'Phulourie (Split Pea Fritters)',
 'Prawn Curry',
 'Qabuli (Central Asian Rice Pilaf)',
 'Seviyan Ji Khirni (Sindhi Vermicelli Pudding)',
 'Sindhi Fried Potatoes',
 'Sweet Lassi',
 'Sweet Mango Lassi',
 'Sylheti Doner Kebab',
 'Sylheti Fried Rice',
 'Sylheti Handesh',
 'Sylheti Rice Pudding',
 'Watalappam (Sri Lankan Coconut Custard)']

In [ ]:

siteb.document_relevant_urls_extraction()

[(tensor([-4.0744e-02,  3.6343e-02, -2.7476e-02,  7.8574e-02, -6.6891e-02,
           1.2443e-02,  1.0935e-02, -4.2737e-02,  7.5238e-02, -4.0938e-02,
           1.6980e-02,  3.5925e-03, -4.5898e-02, -8.7991e-02, -2.6666e-02,
          -1.0692e-01,  1.8051e-02, -4.3807e-02,  1.1212e-02, -1.4118e-01,
          -6.5935e-02,  7.9971e-02,  2.7471e-02,  2.6480e-02, -7.0361e-05,
          -2.6184e-02,  1.2050e-02, -4.1537e-03, -8.0505e-02, -6.5905e-02,
           3.1552e-02, -6.8831e-03,  9.2332e-03,  8.6949e-03,  1.6738e-02,
          -4.3615e-02,  5.5523e-02,  1.0780e-02,  5.8794e-02, -8.5457e-03,
           3.4802e-02, -9.1361e-02,  7.1719e-03,  1.1559e-02,  2.2191e-02,
           5.6710e-02, -4.8959e-02, -8.1062e-03,  6.0982e-02,  3.1864e-03,
          -7.4920e-02, -2.4986e-02, -5.0880e-02, -2.5317e-02,  8.1663e-02,
          -2.0854e-03, -3.8011e-02,  1.9254e-03, -6.5422e-03, -2.6107e-02,
          -3.2650e-02, -2.4600e-02, -5.6503e-02,  4.6292e-02,  3.4448e-03,
           7.7809e-03, -2

In [215]:
sitea=Website("https://en.wikibooks.org/wiki/Cookbook:Baingan_Bartha_(South_Indian_Eggplant_with_Chili)_II")
sitea.extract_all()
sitea.categories

['Medium Difficulty recipes',
 'Recipes',
 'Indian recipes',
 'Recipes using eggplant',
 'Recipes using garlic',
 'Recipes using fresh ginger',
 'Recipes using onion',
 'Recipes using whole cumin',
 'Recipes using turmeric',
 'Recipes using cilantro',
 'Recipes using chile',
 'Recipes using mustard seed']

In [199]:
sitep=Website("https://en.wikibooks.org/wiki/Cookbook:Blender")
sitep.extract_all()

In [232]:
Website.url_title_extraction("https://en.wikipedia.org/wiki/50%25_Rule",type=SiteType.WIKIPEDIA)

'50% Rule'

In [195]:
for con in siteb.content:
    print("------->",con.section_title)
    print(con.section_text)


-------> Introduction
.mw-parser-output .infobox{border:1px solid #a2a9b1;color:black;padding:0.2em;font-size:88%;line-height:1.5em;border-spacing:3px}@media screen{.mw-parser-output .infobox{background-color:#f8f9fa}}@media(max-width:640px){.mw-parser-output .infobox{width:100%}.mw-parser-output .infobox .nowrap{white-space:normal}}@media(min-width:640px){.mw-parser-output .infobox{margin:0.5em 0 0.5em 1em;float:right;clear:right;width:22em}}.mw-parser-output .infobox-header,.mw-parser-output .infobox-label,.mw-parser-output .infobox-above,.mw-parser-output .infobox-full-data,.mw-parser-output .infobox-data,.mw-parser-output .infobox-below,.mw-parser-output .infobox-subheader,.mw-parser-output .infobox-image,.mw-parser-output .infobox-navbar,.mw-parser-output .infobox th,.mw-parser-output .infobox td{vertical-align:top}.mw-parser-output .infobox-label,.mw-parser-output .infobox-data,.mw-parser-output .infobox th,.mw-parser-output .infobox td{text-align:left}.mw-parser-output .infobox 

In [177]:
siteb.references.section_text

'\n\n\n↑ a b c d e f g Davidson, Alan (2014-01-01). Jaine, Tom (ed.). The Oxford Companion to Food. Oxford University Press. doi:10.1093/acref/9780199677337.001.0001. ISBN\xa0978-0-19-967733-7.\n\n↑ a b c d e f g h Kipfer, Barbara Ann (2012-04-11). The Culinarian: A Kitchen Desk Reference. Houghton Mifflin Harcourt. ISBN\xa0978-0-544-18603-3.\n\n↑ a b c d e f g h i Gisslen, Wayne (2015-03-12). Essentials of Professional Cooking, 2nd Edition. Wiley Global Education. ISBN\xa0978-1-119-03072-0.\n\n↑ a b c d e Labensky, Sarah R.; Hause, Alan M.; Martel, Priscilla (2018-01-18). On Cooking: A Textbook of Culinary Fundamentals. Pearson. ISBN\xa0978-0-13-444190-0.\n\n↑ a b c d Foster, Kelli. "Everything You Need to Know About Eggplant".\n\n\n\n\n\n\n\n\n\n\n\n\n\n'

In [180]:
sitec=Website("https://en.wikibooks.org/wiki/Cookbook:Baingan_Bartha_(South_Indian_Eggplant_with_Chili)_II")
sitec.extract_all()

In [181]:
for con in sitec.content:
    print("------->",con.section_title)
    print(con.section_text)

-------> Introduction
.mw-parser-output .infobox{border:1px solid #a2a9b1;color:black;padding:0.2em;font-size:88%;line-height:1.5em;border-spacing:3px}@media screen{.mw-parser-output .infobox{background-color:#f8f9fa}}@media(max-width:640px){.mw-parser-output .infobox{width:100%}.mw-parser-output .infobox .nowrap{white-space:normal}}@media(min-width:640px){.mw-parser-output .infobox{margin:0.5em 0 0.5em 1em;float:right;clear:right;width:22em}}.mw-parser-output .infobox-header,.mw-parser-output .infobox-label,.mw-parser-output .infobox-above,.mw-parser-output .infobox-full-data,.mw-parser-output .infobox-data,.mw-parser-output .infobox-below,.mw-parser-output .infobox-subheader,.mw-parser-output .infobox-image,.mw-parser-output .infobox-navbar,.mw-parser-output .infobox th,.mw-parser-output .infobox td{vertical-align:top}.mw-parser-output .infobox-label,.mw-parser-output .infobox-data,.mw-parser-output .infobox th,.mw-parser-output .infobox td{text-align:left}.mw-parser-output .infobox 

In [70]:
class WikipediaWebsite(Website):
    pass

In [ ]:
class WikiBookWebsite(Website):
    pass

In [ ]:
import wikipediaapi
wiki_wiki = wikipediaapi.Wikipedia(user_agent='Cuisine student project(karan.kamat@postgrad.manchester.ac.uk)', language='en')

page_py = wiki_wiki.page('Python_(programming_language)')

In [ ]:
text = re.sub(r'\[(\d+)\]', '', text)